# E13: fully nested evaluation of the stacked fusions

**Revision plan, 00 log section 51 (written 22 Oct 2026, before this run).** The Limitations disclose that the fusion
meta-learners were fitted on out-of-fold predictions of base models that had seen the evaluation fold, which can make the
**cross-validated** fusion estimates optimistic (the held-out estimates are not affected). This notebook measures that
optimism.

- For each of E7a's **5 repeats** and each **outer fold**, the outer-training patients (about 178) are split into **4 inner
  folds**. In each inner fold the **phase-dropout CNN** (E3 protocol: 40 epochs, inner checkpoint split, per-fold class weights)
  and the **E2 radiomics** recipe are trained on the inner-training patients and predict the inner-validation patients.
- The **CNN + radiomics** and **full-fusion** meta-learners (E2 recipe) are fitted on these inner out-of-fold predictions only,
  and applied to the outer fold's base predictions saved by E7a (base models trained on all outer-training patients).
- Reported per repeat: nested versus non-nested (E7a) pooled out-of-fold macro-AUC; optimism = non-nested − nested; nested
  fusion − branch differences. **The main models and all main results stay unchanged.**

**Gates (the notebook stops if one fails):** E7a's fold assignments match the fold files and their recomputation; the radiomics
recipe on the outer folds reproduces E7a's radiomics out-of-fold predictions; the non-nested fusions recomputed from E7a's
saved base predictions reproduce E7a's saved fusion predictions and per-repeat AUCs.

**Runtime:** **A100 GPU**, about 5–7 hours for all 5 repeats (100 CNN trainings; only predictions are kept, no weights).
**Two sessions in parallel (optional):** open this notebook twice and set `REPEATS_TO_RUN = [0, 1, 2]` in one and `[3, 4]` in
the other; each writes its own folder, and whichever finishes last also writes the combined result over all five repeats.
**Resume:** if a run stops, set `RESUME_RUN_ID` in Cell 1 to the stopped run's time (and keep its `REPEATS_TO_RUN`) and run all
again; finished inner trainings are reused. **When it finishes,** the last cell saves everything to Drive and disconnects the
runtime (`SHUTDOWN_WHEN_DONE = False` keeps it). **If a cell stops with an error,** the error is saved to `E13_error.txt` and the
runtime disconnects (`SHUTDOWN_ON_ERROR = False` keeps it). **Safety:** inputs are only read; outputs go to a new folder
`liver_project/revision_2026-10/E13_<time>/`.

In [ ]:
# Cell 1 — Setup: installs, paths, input checks, output folder, disconnect on error
import os, sys, json, time, random, hashlib, platform, datetime, warnings, subprocess
from pathlib import Path
warnings.filterwarnings('ignore')
RESUME_RUN_ID = None              # to continue a stopped run, put its time here, e.g. '20261022T020304Z'
REPEATS_TO_RUN = [0, 1, 2, 3, 4]  # all five repeats; to use two sessions at once: [0, 1, 2] in one and [3, 4] in the other
SHUTDOWN_ON_ERROR = True          # if any cell stops with an error: save it to Drive (E13_error.txt) and disconnect

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai', 'lightgbm==4.6.0'], check=True)
else:
    ROOT = Path(os.environ['E13_TEST_ROOT'])

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import lightgbm as lgb
from monai.networks.nets import resnet18
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import f_classif
from sklearn.metrics import roc_auc_score

PROJECT_DIR = ROOT / 'liver_project'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E7A_DIR = REVISION_DIR / 'E7a_20261017T191025Z' if IN_COLAB else Path(os.environ['E13_TEST_E7A'])
SEED = 42
P3 = ['p_HCC', 'p_ICC', 'p_CHCC']
NUM_EPOCHS = int(os.environ.get('E13_TEST_EPOCHS', 40))
BATCH_SIZE, LR, WD = 4, 1e-4, 1e-5
N_INNER = 4
INPUTS = {
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e7a_folds': (E7A_DIR / 'E7a_fold_assignments.json', '4eb06167de2c5db223ce09efe23dbbf7dac0ce69e8e812c3be95db4801595011'),
    'e7a_predictions': (E7A_DIR / 'E7a_predictions.csv', 'e74bb57dc49c22b1a3d3ad3bea6dddbe3b4b2428a79876037562b568e17bf751'),
    'e7a_per_repeat': (E7A_DIR / 'E7a_per_repeat.csv', 'f8ccaaa505dacefe1df478bfae2ac3af0e4e5b485283cc13276e08ba65bee413'),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()
problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}'); continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected and IN_COLAB:
        problems.append(f'changed: {path.name}')
tar_path = PROJECT_DIR / 'processed.tar'
if not tar_path.exists() or (IN_COLAB and tar_path.stat().st_size != 402616320):
    problems.append(f'missing or changed size: {tar_path}')
if sorted(set(REPEATS_TO_RUN)) != sorted(REPEATS_TO_RUN) or not set(REPEATS_TO_RUN) <= {0, 1, 2, 3, 4} or not REPEATS_TO_RUN:
    problems.append(f'REPEATS_TO_RUN must list distinct repeats from 0 to 4: {REPEATS_TO_RUN}')
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
WORKERS = 4 if IN_COLAB else 0
import sklearn, monai
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__, 'lightgbm': lgb.__version__,
            'scikit-learn': sklearn.__version__, 'numpy': np.__version__,
            'device': torch.cuda.get_device_name(0) if USE_AMP else 'cpu'}
RUN_ID = RESUME_RUN_ID or datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = REVISION_DIR / f'E13_{RUN_ID}'
OUT_DIR.mkdir(parents=False, exist_ok=bool(RESUME_RUN_ID))
if IN_COLAB and SHUTDOWN_ON_ERROR:
    from IPython import get_ipython
    def _disconnect_on_error(shell, etype, evalue, tb, tb_offset=None):
        shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
        try:
            (OUT_DIR / 'E13_error.txt').write_text(f'{etype.__name__}: {evalue}')
            from google.colab import drive as _drive, runtime as _runtime
            print('\nThe run stopped with the error above; it is saved in Drive (E13_error.txt). The runtime disconnects '
                  'in 60 seconds to save compute units; this output stays visible. Send it for review.')
            _drive.flush_and_unmount()
            time.sleep(60)
            _runtime.unassign()
        except Exception as error:
            print('Could not disconnect automatically:', error)
    get_ipython().set_custom_exc((Exception,), _disconnect_on_error)
print('Versions:', VERSIONS, '| repeats in this run:', REPEATS_TO_RUN, '| output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Crops, labels, E7a's fold assignments (checked) and the inner folds of every outer fold
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
    if len(list(PROCESSED_DIR.glob('*.npz'))) < 278:
        subprocess.run(['cp', str(tar_path), '/content/processed.tar'], check=True)
        subprocess.run(['tar', '-xf', '/content/processed.tar', '-C', '/content/'], check=True)
        os.remove('/content/processed.tar')
else:
    PROCESSED_DIR = Path(os.environ['E13_TEST_PROCESSED'])
split = json.loads(INPUTS['split'][0].read_text())
dev_ids = [str(x) for x in split['dev_ids']]
CT, LABEL = {}, {}
for pid in dev_ids:
    with np.load(PROCESSED_DIR / f'{pid}.npz', allow_pickle=False) as z:
        CT[pid], LABEL[pid] = z['ct'].astype(np.uint8), int(z['label'])
y_dev = np.array([LABEL[p] for p in dev_ids])
REPEATS = {int(r): folds for r, folds in json.loads(INPUTS['e7a_folds'][0].read_text()).items()}
GATE = {}
original = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
same0 = all(sorted(map(str, REPEATS[0][f]['val_ids'])) == sorted(map(str, original[f]['val_ids'])) for f in range(5))
same_new = all(sorted(REPEATS[r][f]['val_ids']) == sorted(dev_ids[i] for i in va)
               for r in range(1, 5) for f, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=r).split(dev_ids, y_dev)))
GATE['e7a_folds_reproduced'] = bool(same0 and same_new)
print('Gate — E7a fold assignments equal the fold files (repeat 0) and their recomputation (repeats 1-4):', GATE['e7a_folds_reproduced'])
if not GATE['e7a_folds_reproduced']:
    raise RuntimeError('STOP — E7a fold assignments do not match. Send this output.')
for r in REPEATS:
    for s in REPEATS[r]:
        s['train_ids'], s['val_ids'] = [str(p) for p in s['train_ids']], [str(p) for p in s['val_ids']]
INNER = {}                                       # (repeat, outer fold) -> [(inner_train, inner_val)] * 4
for r in REPEATS_TO_RUN:
    for k in range(5):
        outer_train = list(REPEATS[r][k]['train_ids'])
        skf = StratifiedKFold(N_INNER, shuffle=True, random_state=1000 + 10 * r + k)
        INNER[(r, k)] = [([outer_train[i] for i in tr], [outer_train[i] for i in va])
                         for tr, va in skf.split(outer_train, [LABEL[p] for p in outer_train])]
        vals = sum((va for _, va in INNER[(r, k)]), [])
        assert sorted(vals) == sorted(outer_train) and len(set(vals)) == len(vals), (r, k)
        assert not set(outer_train) & set(REPEATS[r][k]['val_ids']), (r, k)
(OUT_DIR / 'E13_inner_assignments.json').write_text(json.dumps({f'{r}_{k}': v for (r, k), v in INNER.items()}))
print('Inner folds:', {f'{r}_{k}': [len(va) for _, va in v] for (r, k), v in list(INNER.items())[:3]}, '...',
      f'({len(INNER)} outer folds x {N_INNER} inner folds = {len(INNER) * N_INNER} CNN trainings)')

In [ ]:
# Cell 3 — E3 phase-dropout training code (E7a's, unchanged) and the inner-fold training function
def zscore4(x4):
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        out[c][mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def augment(vol4):
    if random.random() < 0.5:
        vol4 = vol4[:, ::-1].copy()
    if random.random() < 0.5:
        vol4 = vol4[:, :, ::-1].copy()
    if random.random() < 0.5:
        for c in range(vol4.shape[0]):
            scale = 1.0 + (random.random() - 0.5) * 0.2
            shift = (random.random() - 0.5) * 0.2
            mask = vol4[c] != 0
            vol4[c][mask] = vol4[c][mask] * scale + shift
    return vol4

def sample_kept_phases():
    if random.random() < 0.5:
        return (0, 1, 2, 3)
    while True:
        kept = tuple(c for c in range(4) if random.random() >= 0.5)
        if kept:
            return kept

def keep_only(vol4, kept):
    out = vol4.copy()
    for c in range(4):
        if c not in kept:
            out[c] = 0.0
    return out

class CropDataset(Dataset):
    def __init__(self, ids, train=False):
        self.ids, self.train = list(ids), train
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, index):
        pid = self.ids[index]
        vol4 = zscore4(CT[pid].astype(np.float32))
        if self.train:
            vol4 = augment(vol4)
            kept = sample_kept_phases()
            if kept != (0, 1, 2, 3):
                vol4 = keep_only(vol4, kept)
        return torch.from_numpy(np.ascontiguousarray(vol4, dtype=np.float32)), LABEL[pid], pid

def seed_worker(worker_id):
    np.random.seed(torch.initial_seed() % 2 ** 32)

def make_model():
    return resnet18(spatial_dims=3, n_input_channels=4, num_classes=3, feed_forward=True)

def class_weights_for(ids):
    counts = np.bincount([LABEL[p] for p in ids], minlength=3).astype(np.float32)
    return torch.from_numpy(counts.sum() / (3 * counts.clip(min=1))).float()

def macro_auc(y, p):
    return float(roc_auc_score(np.asarray(y), np.asarray(p), multi_class='ovr', average='macro'))

@torch.no_grad()
def predict(model, ids, batch_size=8):
    model.eval()
    loader = DataLoader(CropDataset(ids), batch_size=batch_size, shuffle=False, num_workers=2 if IN_COLAB else 0, pin_memory=USE_AMP)
    probs, pids = [], []
    for x, _, batch_ids in loader:
        probs.append(torch.softmax(model(x.to(DEVICE, non_blocking=True)).float().cpu(), dim=1).numpy())
        pids.extend(batch_ids)
    return pd.DataFrame(np.concatenate(probs), index=pids, columns=P3)

INNER_DIR = OUT_DIR / 'inner_cnn_predictions'
INNER_DIR.mkdir(exist_ok=True)
def train_inner(r, k, j):
    """E7a's train_fold on the inner-training patients of (repeat r, outer fold k, inner fold j); keeps only the
    predictions for the inner-validation patients and the training log (no weights)."""
    path = INNER_DIR / f'cnn_r{r}_k{k}_j{j}.csv'
    if path.exists():
        return 'reused'
    inner_train, inner_val = INNER[(r, k)][j]
    seed = 2000 + 100 * r + 10 * k + j
    fit_ids, ckpt_ids = train_test_split(inner_train, test_size=0.2, random_state=seed, stratify=[LABEL[p] for p in inner_train])
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if USE_AMP:
        torch.cuda.manual_seed_all(seed)
    loader = DataLoader(CropDataset(fit_ids, train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS,
                        pin_memory=USE_AMP, drop_last=True, persistent_workers=WORKERS > 0, worker_init_fn=seed_worker)
    model = make_model().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = nn.CrossEntropyLoss(weight=class_weights_for(fit_ids).to(DEVICE))
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    y_ckpt = np.array([LABEL[p] for p in ckpt_ids])
    best_auc, best_state, best_epoch = -1.0, None, -1
    t0 = time.time()
    for epoch in range(NUM_EPOCHS):
        model.train()
        for x, y, _ in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
        auc = macro_auc(y_ckpt, predict(model, ckpt_ids).loc[ckpt_ids].to_numpy())
        if auc > best_auc:
            best_auc, best_epoch = auc, epoch
            best_state = {key: v.detach().cpu().clone() for key, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    out = predict(model, inner_val).loc[inner_val]
    out.insert(0, 'patient_id', out.index)
    for name, value in [('repeat', r), ('outer_fold', k), ('inner_fold', j), ('best_epoch', best_epoch),
                        ('best_checkpoint_auc', best_auc), ('n_fit', len(fit_ids)), ('minutes', (time.time() - t0) / 60)]:
        out[name] = value
    tmp = path.with_suffix('.tmp')
    out.to_csv(tmp, index=False)
    os.replace(tmp, path)
    del model
    if USE_AMP:
        torch.cuda.empty_cache()
    return f'checkpoint AUC {best_auc:.3f} at epoch {best_epoch} ({(time.time() - t0) / 60:.1f} min)'

In [ ]:
# Cell 4 — Train the inner-fold CNNs (resumable: finished inner trainings are reused)
JOBS = [(r, k, j) for r in REPEATS_TO_RUN for k in range(5) for j in range(N_INNER)]
t0, trained = time.time(), 0
for n, (r, k, j) in enumerate(JOBS, start=1):
    status = train_inner(r, k, j)
    if status != 'reused':
        trained += 1
        rate = (time.time() - t0) / trained
        print(f'  [{n}/{len(JOBS)}] repeat {r}, outer fold {k}, inner fold {j}: {status} | about '
              f'{rate * (len(JOBS) - n) / 3600:.1f} h left', flush=True)
print(f'Inner CNN trainings done: {trained} trained now, {len(JOBS) - trained} reused ({(time.time() - t0) / 60:.1f} min)')

In [ ]:
# Cell 5 — Radiomics (E2 recipe, as E7a): gate on the outer folds, then the inner folds
K_TOP_FEATURES, CORR_THRESHOLD = 30, 0.95
def select_features_train(X_tr, y_tr, feature_names, k=K_TOP_FEATURES, corr_thresh=CORR_THRESHOLD):   # Week 5 / E2, unchanged
    stds = np.nanstd(X_tr, axis=0)
    keep = stds > 1e-10
    X_kept = X_tr[:, keep]
    kept_idx_global = np.where(keep)[0]
    X_nan_to_num = np.nan_to_num(X_kept, nan=np.nanmedian(X_kept))
    corr = np.abs(np.corrcoef(X_nan_to_num.T))
    remove = np.zeros(corr.shape[0], dtype=bool)
    for i in range(corr.shape[0]):
        if remove[i]:
            continue
        for j in range(i + 1, corr.shape[0]):
            if remove[j]:
                continue
            if corr[i, j] > corr_thresh:
                remove[j] = True
    kept_idx_after_corr = kept_idx_global[~remove]
    X_for_f = SimpleImputer(strategy='median').fit_transform(X_tr[:, kept_idx_after_corr])
    F, _ = f_classif(X_for_f, y_tr)
    order = np.argsort(F)[::-1]
    final_idx = kept_idx_after_corr[order[:min(k, len(order))]]
    return final_idx, [feature_names[i] for i in final_idx]

def radiomics_fit_predict(train_ids, predict_ids):  # E7a radiomics_repeat, one fold (rows in the feature table's order, as E7a)
    tr = internal[internal['patient_id'].isin(set(train_ids))]
    X_tr, y_tr = tr[POOL].to_numpy(dtype=np.float32), tr['label'].to_numpy()
    idx, _ = select_features_train(X_tr, y_tr, POOL)
    imp = SimpleImputer(strategy='median').fit(X_tr[:, idx])
    sc = StandardScaler().fit(imp.transform(X_tr[:, idx]))
    clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, num_leaves=15, min_child_samples=5,
                             reg_alpha=0.1, reg_lambda=0.1, class_weight='balanced', random_state=42, verbose=-1)
    clf.fit(sc.transform(imp.transform(X_tr[:, idx])), y_tr)
    X_p = FEAT.loc[predict_ids, POOL].to_numpy(dtype=np.float32)
    return pd.DataFrame(clf.predict_proba(sc.transform(imp.transform(X_p[:, idx]))), index=predict_ids, columns=P3)

internal = pd.read_parquet(INPUTS['radiomics_features'][0])
internal['patient_id'] = internal['patient_id'].astype(str)
POOL = [c for c in internal.columns if c not in ('patient_id', 'label') and '_original_shape_' not in c]   # E2: 372 features
FEAT = internal.set_index('patient_id')
E7A = pd.read_csv(INPUTS['e7a_predictions'][0], dtype={'patient_id': str})
def e7a(r, model):
    t = E7A[(E7A['repeat'] == r) & (E7A['model'] == model) & (E7A['split'] == 'oof')].set_index('patient_id')
    return t.loc[dev_ids, P3]
gate = {}
for r in REPEATS_TO_RUN:                         # gate: the outer folds reproduce E7a's radiomics out-of-fold predictions
    outer = pd.concat([radiomics_fit_predict(s['train_ids'], s['val_ids']) for s in REPEATS[r]]).loc[dev_ids]
    gate[r] = float(np.abs(outer.to_numpy() - e7a(r, 'radiomics_noshape').to_numpy()).max())
GATE['radiomics_outer_vs_E7a'] = gate
print('Gate — radiomics on the outer folds vs E7a (max difference per repeat):', gate)
if IN_COLAB and max(gate.values()) > 1e-6:
    raise RuntimeError('STOP — the radiomics recipe does not reproduce E7a. Send this output.')
RAD_INNER = {}                                   # (r, k) -> inner out-of-fold radiomics predictions of the outer-training patients
for (r, k), folds in INNER.items():
    RAD_INNER[(r, k)] = pd.concat([radiomics_fit_predict(tr, va) for tr, va in folds])
CNN_INNER, LOG = {}, []
for (r, k), folds in INNER.items():
    parts = [pd.read_csv(INNER_DIR / f'cnn_r{r}_k{k}_j{j}.csv', dtype={'patient_id': str}) for j in range(N_INNER)]
    LOG += [p.drop(columns=['patient_id'] + P3).iloc[0].to_dict() for p in parts]
    CNN_INNER[(r, k)] = pd.concat(parts).set_index('patient_id')[P3]
    assert sorted(CNN_INNER[(r, k)].index) == sorted(REPEATS[r][k]['train_ids']), (r, k)
LOG = pd.DataFrame(LOG)
print('Inner radiomics fitted for', len(RAD_INNER), 'outer folds | inner CNN predictions read:', len(LOG), 'trainings')

In [ ]:
# Cell 6 — Meta-learners: non-nested (E7a, recomputed as a gate) and fully nested
W4, W5 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC'], ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
FUSIONS = {'cnn_plus_radiomics': W4 + W5, 'full_fusion': W4 + W5 + ['age', 'gender']}
META = pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[dev_ids]
def prepare(X):                                  # E2 / E7a, unchanged
    X = X.astype(np.float32)
    X[X <= -1] = np.nan
    return X
def fit_meta(frame, columns):                    # E7a meta_cv, one fold
    X = prepare(frame[columns].to_numpy())
    imp = SimpleImputer(strategy='median').fit(X)
    sc = StandardScaler().fit(imp.transform(X))
    m = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED).fit(sc.transform(imp.transform(X)), frame['label'].astype(int))
    return lambda F: m.predict_proba(sc.transform(imp.transform(prepare(F[columns].to_numpy()))))
def base_frame(ids, cnn, rad):
    f = META.loc[ids, ['label', 'age', 'gender']].copy()
    f[W4], f[W5] = cnn.loc[ids].to_numpy(), rad.loc[ids].to_numpy()
    return f
NONNESTED, NESTED, check = {}, {}, {}
for r in REPEATS_TO_RUN:
    outer_cnn, outer_rad = e7a(r, 'phase_dropout_CNN'), e7a(r, 'radiomics_noshape')
    for name, cols in FUSIONS.items():
        plain, nested = pd.DataFrame(index=dev_ids, columns=P3, dtype=float), pd.DataFrame(index=dev_ids, columns=P3, dtype=float)
        for k, s in enumerate(REPEATS[r]):
            train_set = set(s['train_ids'])
            tr, va = [p for p in dev_ids if p in train_set], s['val_ids']        # E7a's row order
            plain.loc[va] = fit_meta(base_frame(tr, outer_cnn, outer_rad), cols)(base_frame(va, outer_cnn, outer_rad))     # E7a
            nested.loc[va] = fit_meta(base_frame(tr, CNN_INNER[(r, k)], RAD_INNER[(r, k)]), cols)(base_frame(va, outer_cnn, outer_rad))
        NONNESTED[(r, name)], NESTED[(r, name)] = plain, nested
        check[f'{r}_{name}'] = float(np.abs(plain.to_numpy() - e7a(r, name).to_numpy()).max())
GATE['nonnested_fusions_vs_E7a'] = check
print('Gate — non-nested fusions recomputed vs E7a saved predictions (max difference):', check)
if IN_COLAB and max(check.values()) > 1e-6:
    raise RuntimeError('STOP — the non-nested fusions do not reproduce E7a. Send this output.')

In [ ]:
# Cell 7 — Results: nested versus non-nested per repeat, optimism, fusion versus branches; saved outputs; combined result
E7A_PR = pd.read_csv(INPUTS['e7a_per_repeat'][0]).set_index(['repeat', 'model'])
rows = []
for r in REPEATS_TO_RUN:
    branch = {b: macro_auc(y_dev, e7a(r, b).to_numpy()) for b in ['phase_dropout_CNN', 'radiomics_noshape']}
    for name in FUSIONS:
        fold_of = {p: f for f, s in enumerate(REPEATS[r]) for p in s['val_ids']}
        def fold_mean(frame):
            return float(np.mean([macro_auc([LABEL[p] for p in s['val_ids']], frame.loc[s['val_ids']].to_numpy()) for s in REPEATS[r]]))
        plain, nested = macro_auc(y_dev, NONNESTED[(r, name)].to_numpy()), macro_auc(y_dev, NESTED[(r, name)].to_numpy())
        rows.append({'repeat': r, 'model': name, 'oof_nonnested': plain, 'oof_nested': nested, 'optimism_nonnested_minus_nested': plain - nested,
                     'fold_mean_nonnested': fold_mean(NONNESTED[(r, name)]), 'fold_mean_nested': fold_mean(NESTED[(r, name)]),
                     'E7a_saved_oof_pooled': float(E7A_PR.loc[(r, name), 'oof_pooled_auc']),
                     'branch_CNN': branch['phase_dropout_CNN'], 'branch_radiomics': branch['radiomics_noshape'],
                     'nested_minus_CNN': nested - branch['phase_dropout_CNN'], 'nested_minus_radiomics': nested - branch['radiomics_noshape'],
                     'nonnested_minus_CNN': plain - branch['phase_dropout_CNN'], 'nonnested_minus_radiomics': plain - branch['radiomics_noshape']})
PER_REPEAT = pd.DataFrame(rows)
GATE['nonnested_auc_vs_E7a_per_repeat'] = float((PER_REPEAT['oof_nonnested'] - PER_REPEAT['E7a_saved_oof_pooled']).abs().max())
if IN_COLAB and GATE['nonnested_auc_vs_E7a_per_repeat'] > 1e-9:
    raise RuntimeError('STOP — the recomputed non-nested AUCs differ from E7a_per_repeat. Send this output.')

def summarize(per_repeat):
    out = []
    for name, g in per_repeat.groupby('model', sort=False):
        out.append({'model': name, 'repeats': len(g), 'oof_nonnested_mean': g['oof_nonnested'].mean(), 'oof_nested_mean': g['oof_nested'].mean(),
                    'oof_nested_sd': g['oof_nested'].std(ddof=1) if len(g) > 1 else np.nan,
                    'optimism_mean': g['optimism_nonnested_minus_nested'].mean(), 'optimism_min': g['optimism_nonnested_minus_nested'].min(),
                    'optimism_max': g['optimism_nonnested_minus_nested'].max(),
                    'repeats_nested_above_CNN': int((g['nested_minus_CNN'] > 0).sum()),
                    'repeats_nested_above_radiomics': int((g['nested_minus_radiomics'] > 0).sum()),
                    'nested_minus_CNN_mean': g['nested_minus_CNN'].mean(), 'nested_minus_radiomics_mean': g['nested_minus_radiomics'].mean()})
    return pd.DataFrame(out)
SUMMARY = summarize(PER_REPEAT)

long = []
for (r, name), frame in NESTED.items():
    t = frame.copy(); t.insert(0, 'patient_id', t.index); t.insert(0, 'model', name); t.insert(0, 'repeat', r); long.append(t)
pd.concat(long, ignore_index=True).to_csv(OUT_DIR / 'E13_nested_oof_predictions.csv', index=False)
pd.concat([f.assign(repeat=r, outer_fold=k).rename_axis('patient_id').reset_index() for (r, k), f in RAD_INNER.items()]).to_csv(
    OUT_DIR / 'E13_inner_radiomics_predictions.csv', index=False)
LOG.to_csv(OUT_DIR / 'E13_inner_training_log.csv', index=False)
PER_REPEAT.to_csv(OUT_DIR / 'E13_per_repeat.csv', index=False)
SUMMARY.to_csv(OUT_DIR / 'E13_summary_this_run.csv', index=False)
PROTOCOL = '00 log section 51, E13 (written 22 Oct 2026, before the run); the main models are unchanged'
summary = {'run_id': RUN_ID, 'notebook': 'R16_E13_nested_stacking.ipynb', 'versions': VERSIONS, 'input_sha256': input_hashes, 'gates': GATE,
           'epochs': NUM_EPOCHS, 'repeats_in_this_run': REPEATS_TO_RUN, 'inner_folds': N_INNER, 'protocol': PROTOCOL}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))

# ---- combined result over all E13 runs of this protocol (written by whichever run completes the five repeats) ----
found = []
for folder in sorted(REVISION_DIR.glob('E13_*')):
    try:
        info = json.loads((folder / 'summary.json').read_text())
        if info.get('protocol') == PROTOCOL and info.get('epochs') == NUM_EPOCHS and (folder / 'E13_per_repeat.csv').exists():
            found.append(pd.read_csv(folder / 'E13_per_repeat.csv').assign(run=folder.name))
    except Exception:
        continue
ALL = pd.concat(found, ignore_index=True).sort_values('run').drop_duplicates(['repeat', 'model'], keep='last') if found else PER_REPEAT
COMBINED = None
if sorted(ALL['repeat'].unique()) == [0, 1, 2, 3, 4]:
    COMBINED = summarize(ALL.sort_values(['repeat', 'model']))
    COMBINED.to_csv(OUT_DIR / 'E13_combined_summary.csv', index=False)
    ALL.to_csv(OUT_DIR / 'E13_combined_per_repeat.csv', index=False)
    RULES = {}
    for _, row in COMBINED.iterrows():
        RULES[row['model']] = {
            'optimism': f"mean {row['optimism_mean']:+.3f} (range {row['optimism_min']:+.3f} to {row['optimism_max']:+.3f}) over 5 repeats",
            'tables_need_nested_values': bool(row['optimism_mean'] > 0.01),
            'above_both_branches_in_all_5_repeats_nested': bool(row['repeats_nested_above_CNN'] == 5 and row['repeats_nested_above_radiomics'] == 5)}
    summary['combined'] = {'runs': sorted(ALL['run'].unique().tolist()), 'rules': RULES}
    (OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)} for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E13_{RUN_ID} | repeats {REPEATS_TO_RUN} | epochs {NUM_EPOCHS} | gates: {GATE}')
print('\n--- per repeat (this run) ---'); print(PER_REPEAT.drop(columns=['E7a_saved_oof_pooled']).round(4).to_string(index=False))
print('\n--- summary (this run) ---'); print(SUMMARY.round(4).to_string(index=False))
if COMBINED is not None:
    print('\n--- COMBINED over all five repeats (runs: ' + ', '.join(sorted(ALL['run'].unique())) + ') ---')
    print(COMBINED.round(4).to_string(index=False))
    print('\nPre-specified statements:', json.dumps(summary['combined']['rules'], indent=1))
else:
    print('\nThe combined result over all five repeats will be written by the run that completes them.')
print('==============================================================')

In [ ]:
# Cell 8 — Finish: make sure every result file has reached Google Drive, then disconnect the runtime (stops compute units)
SHUTDOWN_WHEN_DONE = True     # set to False to keep the runtime after the run
if IN_COLAB and SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print('All files are saved in Drive. The runtime disconnects in 30 seconds; the outputs above stay visible on this page.')
    time.sleep(30)
    runtime.unassign()